# Travel Planner Agent

In [ ]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
import os
from IPython.display import Markdown, display
from datetime import datetime
load_dotenv(override=True)
import gradio as gr
import asyncio

from gr_app import extract_profile, is_ready_to_plan, message_gpt

In [ ]:
MODEL1 = "gpt-oss:20b"
MODEL2 = "llama3.2:latest"

## MCP Servers

### Web Surfer MCP Tool

In [ ]:
web_tool_instructions = "You are able to surf the web for flight information, including prices, availability, and schedules. Use this tool to find the best flight options for the user's travel plans. " \
"Always provide accurate and up-to-date information when using this tool." \
"You can also find information about hotels, car rentals, and other travel-related services. Use this tool to help the user plan their trip effectively."

In [ ]:
playwright_params = {"command": "npx","args": [ "@playwright/mcp@latest"]}

async def web_tool_response_handler(request):
    async with MCPServerStdio(params=playwright_params, client_session_timeout_seconds=30) as mcp_server:
        agent = Agent(name="web_agent", instructions=web_tool_instructions, model=MODEL2, mcp_servers=[mcp_server])
        with trace("conversation"):
            result = await Runner.run(agent, request)
            return result
        # mcp_tools = await server.list_tools()

### Uber Tool

In [ ]:
instructions = "Given users location and destination, you can use this tool to give an estimated cost of a ride for the user. Always provide accurate and up-to-date information when using this tool."

In [ ]:
env = {
    "UBER_CLIENT_ID": os.getenv("UBER_CLIENT_ID"), 
    "UBER_CLIENT_SECRET": os.getenv("UBER_CLIENT_SECRET"), 
    "UBER_ENVIRONMENT": os.getenv("UBER_ENVIRONMENT", "sandbox"),
    "UBER_REDIRECT_URI": os.getenv("UBER_REDIRECT_URI", "http://localhost:3000/callback")
}

uber_params = {"command": "npx", "args": ["-y", "mcp-uber"], "env": env}

async def uber_tool_response_handler(request):
    async with MCPServerStdio(params=uber_params, client_session_timeout_seconds=30) as mcp_server:
        agent = Agent(name="uber_agent", instructions=instructions, model=MODEL2, mcp_servers=[mcp_server])
        with trace("conversation"):
            result = await Runner.run(agent, request)
            return result

### File I/O MCP Tool

In [ ]:
instructions = """
You are a travel organiser assistant. 
You have access to three tools: a database tool for storing and retrieving travel information, a web search tool for finding flight and hotel information, and a ride estimation tool for calculating the cost of rides. 
Use these tools to help the user plan their trip effectively. 
Always provide accurate and up-to-date information when using these tools. When done, create a file with the itinerary for the user based on the information you have gathered and the user's preferences.
"""

In [ ]:
pwd = os.path.abspath(os.path.join(os.getcwd(), "."))
files_params = {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-filesystem", pwd]}


async def converse(prompt):
    async with MCPServerStdio(params=files_params, client_session_timeout_seconds=60) as mcp_server_files:
        async with MCPServerStdio(params=playwright_params, client_session_timeout_seconds=60) as mcp_server_browser:
            agent = Agent(
                name="travel_agent", 
                instructions=instructions, 
                model="gpt-oss:20b",
                mcp_servers=[mcp_server_files, mcp_server_browser]
                )
            with trace("investigate"):
                result = await Runner.run(agent, prompt)
                print(result.final_output)

In [ ]:
async def run_planning_agents(profile):
    results = {}

    web_result = await web_tool_response_handler(
        f"""
        Find best flights and hotels for:
        {profile}
        """
    )
    results["web"] = web_result.final_output

    uber_result = await uber_tool_response_handler(
        f"""
        Estimate transport cost from {profile['origin']} 
        to {profile['destination']}
        """
    )
    results["uber"] = uber_result.final_output

    await converse(
        f"""
        Create a detailed travel itinerary using:
        {profile}
        Flights/Hotels: {results['web']}
        Transport: {results['uber']}
        """
    )

    return results

In [ ]:
planner_instructions = """
You are an autonomous travel planner agent.

Your job:
Plan a complete trip for the user.

You have access to tools:
1. Database → store/retrieve user info
2. Web → find flights, hotels, experiences
3. Uber → estimate transport costs
4. Filesystem → write final itinerary

Rules:
- First ensure you understand the user's trip
- Store important info in the database
- Use web tool to find flights and hotels
- Decide best transport (flight, uber, etc.)
- Combine everything into a plan
- Finally, create a detailed itinerary file

You can call tools multiple times.
Think step-by-step before acting.
Stop only when the plan is complete.
"""

In [ ]:
async def run_agentic_planner(user_request):
    async with MCPServerStdio(params=playwright_params) as web:
        async with MCPServerStdio(params=uber_params) as uber:
            async with MCPServerStdio(params=files_params) as files:

                agent = Agent(
                    name="planner_agent",
                    instructions=planner_instructions,
                    model=MODEL1,
                    mcp_servers=[web, uber, files]
                )

                with trace("agentic-planning"):
                    result = await Runner.run(agent, user_request)
                    return result

In [ ]:
def build_profile_prompt(profile):
    return f"""
Plan a full trip using this information:

{profile}

Tasks:
- Find cheapest flights
- Suggest best transport
- Find hotels
- Create itinerary
- Save itinerary to file
"""

In [ ]:
def travel_agent(user_input, history, state):

    # Step 1: normal chat (intake)
    response = message_gpt(user_input, history)

    # Step 2: detect readiness
    if "yes" in user_input.lower() and not state.get("started"):

        state["started"] = True

        profile = extract_profile(history)

        result = asyncio.run(
            run_agentic_planner(build_profile_prompt(profile))
        )

        response = f"🚀 Planning complete:\n\n{result.final_output}"

    return response, state

In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("# 🌍 AI Travel Advisor")

    chatbot = gr.Chatbot(type="messages")
    msg = gr.Textbox()

    state = gr.State({"started": False})

    def respond(user_input, history, state):
        response, state = travel_agent(user_input, history, state)
        history.append({"role": "user", "content": user_input})
        history.append({"role": "assistant", "content": response})
        return "", history, state

    msg.submit(respond, [msg, chatbot, state], [msg, chatbot, state])

demo.launch() 